[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline.ipynb)

# 01. 상품명으로 카테고리 맞히기 — 텍스트 분류 기준선

**"한결식품 얼큰 컵라면 110g"이라는 글자만 보고 이 상품이 `라면류`인지 `즉석밥/간편식`인지 맞히는 문제**입니다.
AICE Professional 샘플문항의 Text 문제(가공식품 카테고리 분류)와 같은 형태이고, 같은 방식으로 채점됩니다.

## 이 장을 배우는 이유

지금까지 이 저장소에서 다룬 데이터는 **숫자와 범주**였습니다
([tabular-ml-practice](https://github.com/karzit/temp/blob/master/notebooks/tabular-ml-practice/README.md)).
표 데이터는 `fare`, `age`처럼 컬럼 하나가 곧 [피처](https://github.com/karzit/temp/blob/master/glossary.md#feature) 하나였습니다.
**텍스트에는 그런 컬럼이 없습니다.** 입력은 `상품명` 하나뿐이고, 그 안에서 피처를 직접 만들어내야 합니다.
이 노트북의 절반은 "글자를 어떻게 숫자로 바꾸는가"에 대한 이야기입니다.

## 이 노트북의 구성

| 절 | 내용 | 왜 하는가 |
|---|---|---|
| 1~2 | 문제 파악, 데이터 관찰 | 결측·중복·클래스 불균형을 먼저 확인 |
| 3 | 기준선 만들기 | "정확도 70%"가 잘한 건지 판단할 기준 |
| 4 | 텍스트를 숫자로 — BoW와 [TF-IDF](https://github.com/karzit/temp/blob/master/glossary.md#tfidf) | 텍스트 분류의 핵심 |
| 5~7 | 첫 모델, 전처리 실험, 문자 n-gram | 성능을 올리는 방법과 **올리지 못하는 방법** |
| 8~10 | 모델 비교, 평가 지표, 오분류 분석 | 정확도 숫자 하나로 끝내지 않기 |
| 11 | 제출 파일 만들기 | 시험 형식 그대로 |

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행**하세요. 뒤 셀은 앞 셀에서 만든 변수를 씁니다
- 본문에 적힌 숫자는 **여러분이 실행한 결과와 소수점 이하가 다를 수 있습니다**
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요
- 에러가 나면 [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 보세요
- **소요 시간 40분쯤**. 오래 걸리는 셀은 없습니다(가장 무거운 8절 모델 비교도 10초 안팎)

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 대응 |
|---|---|---|
| `FileNotFoundError: 02_train.csv` | 데이터 준비 셀을 건너뜀 | 아래 첫 코드 셀부터 실행 |
| 그래프의 한글이 네모(□)로 | 한글 폰트 없음 | `koreanize-matplotlib` 설치 셀 실행 후 **런타임 재시작** |
| `ValueError: empty vocabulary` | 전처리가 텍스트를 전부 지움 | 정제 함수가 한글까지 지우지 않는지 확인 |

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

BASE_URL = "https://raw.githubusercontent.com/karzit/temp/master/notebooks/text-classification-practice/data"

if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib seaborn koreanize-matplotlib joblib
    for _f in ["02_train.csv", "02_test_x.csv", "02_test_y.csv"]:
        !wget -q -O {_f} {BASE_URL}/{_f}
    DATA_DIR = "."
else:
    # 저장소를 클론했다면 이 노트북(notebooks/text-classification-practice/01_text_baseline/)
    # 기준으로 ../data 에 csv가 있습니다.
    DATA_DIR = os.path.join("..", "data") if os.path.isdir(os.path.join("..", "data")) else "."

print("데이터 경로:", os.path.abspath(DATA_DIR))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    import koreanize_matplotlib  # noqa: F401  (import만 해도 한글 폰트가 잡힙니다)
except ImportError:
    print("koreanize-matplotlib이 없어 그래프의 한글이 깨질 수 있습니다.")

pd.set_option("display.max_colwidth", 40)
RANDOM_STATE = 42

---

## 1. 문제부터 정확히 읽는다

시험 문제지에 적힌 내용을 표로 옮기면 이렇습니다. **채점되는 것은 모델의 우아함이 아니라 이 표를 지켰는지**입니다.

| 항목 | 내용 |
|---|---|
| 입력 | `상품명` (가공식품 상품명 문자열) |
| 출력 | `카테고리` (가공식품 분류) |
| 훈련 데이터 | `02_train.csv` — 상품명 + 카테고리 |
| 예측 대상 | `02_test_x.csv` — 상품명만 있고 **정답 없음** |
| 달성 목표 | 테스트 정확도 **70% 이상** (채점 통과선은 63%) |
| 제출물 | 모델 파일, 예측 결과 csv, 답안 노트북 |

> **이 연습 데이터에 대하여.** 실제 시험 데이터는 공개되지 않으므로, 같은 구조(`상품명` → `카테고리`)를
> 규칙으로 합성한 데이터를 씁니다. 생성 규칙은
> [`data/make_dataset.py`](https://github.com/karzit/temp/blob/master/notebooks/text-classification-practice/data/make_dataset.py)에
> 그대로 있습니다. 결측·중복·불균형·라벨 오류를 **일부러** 섞어 두었습니다.
> 브랜드명은 실제 상표를 피하려고 지어낸 이름입니다.
>
> 시험에는 없는 파일이 하나 더 있습니다 — `02_test_y.csv`(테스트 정답). **11절에서 자가 채점할 때만** 씁니다.
> 그전에 열어보면 연습이 되지 않습니다.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "02_train.csv"))
test_x = pd.read_csv(os.path.join(DATA_DIR, "02_test_x.csv"))

print("train:", train.shape, " test_x:", test_x.shape)
train.head(10)

---

## 2. 데이터를 먼저 본다

모델을 짜기 전에 **데이터가 어떻게 생겼는지** 확인합니다. 표 데이터에서 `info()`/`describe()`로 했던 일을
텍스트에서도 똑같이 합니다. 볼 것은 세 가지입니다.

1. **결측과 중복** — 텍스트에서도 빈 값은 그대로 에러가 됩니다
2. **클래스 분포** — 몇 대 몇으로 치우쳐 있는가
3. **길이** — 상품명이 몇 글자, 몇 단어짜리인가 (02번에서 시퀀스 길이를 정할 때 씁니다)

In [ ]:
train.info()
print("\n결측:", train["상품명"].isna().sum(), "건")
print("완전 중복 행:", train.duplicated().sum(), "건")

In [ ]:
counts = train["카테고리"].value_counts()
print(counts)
print("\n가장 많은 카테고리의 비율: %.3f" % (counts.iloc[0] / counts.sum()))

plt.figure(figsize=(8, 4))
sns.barplot(x=counts.values, y=counts.index, color="steelblue")
plt.title("카테고리별 건수")
plt.xlabel("건수")
plt.tight_layout()
plt.show()

**관찰 세 가지.**

- **결측 10건, 완전 중복 79건.** 상품명이 비어 있으면 벡터화 단계에서 바로 에러가 납니다. 중복은
  같은 상품이 여러 번 학습되어 그쪽으로 모델이 기울게 만듭니다
- **클래스가 불균형합니다.** 가장 많은 `라면류`가 약 18%, 가장 적은 `시리얼/영양바`는 3% 남짓입니다.
  뒤에서 평가 지표를 고를 때 이 사실이 중요해집니다
- 상품명은 짧습니다. 문장이 아니라 **단어 몇 개를 이어 붙인 형태**입니다

먼저 결측과 중복을 정리합니다. **텍스트에서 결측치 "대체"는 의미가 없습니다.** 표 데이터라면 평균값으로
채울 수 있지만, 상품명을 평균값으로 채울 수는 없습니다. 상품명이 없으면 **아무 정보도 없는 행**이므로 버립니다.

In [ ]:
before = len(train)
train = train.dropna(subset=["상품명"]).drop_duplicates().reset_index(drop=True)
print(f"{before} → {len(train)}행")

길이 = pd.DataFrame({
    "글자수": train["상품명"].str.len(),
    "단어수": train["상품명"].str.split().str.len(),
})
print(길이.describe().round(1))

> **`drop_duplicates()`의 범위에 주의하세요.** 위에서는 기본값(모든 컬럼)이라 *상품명과 카테고리가 모두 같은*
> 행만 지웁니다. 만약 `subset=["상품명"]`을 준다면 **같은 상품명에 다른 카테고리가 붙은 행**까지 하나만 남기고
> 지워집니다. 라벨 오류를 조용히 정리해버리는 셈이라, 실무에서는 그런 행이 몇 건인지 먼저 확인하는 편이 낫습니다
> (연습 문제 1번).

---

## 3. 기준선 — "정확도 70%"는 잘한 걸까?

**모델 없이 얻을 수 있는 성능을 먼저 계산합니다.** 이 숫자를 넘지 못하면 그 모델은 존재할 이유가 없습니다.
가장 단순한 전략은 **무조건 가장 많은 카테고리로 찍기**입니다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X = train["상품명"]
y = train["카테고리"]

# stratify=y : 분할 후에도 카테고리 비율이 유지되도록 합니다. 소수 카테고리가 한쪽에만 몰리는 것을 막습니다.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("학습", len(X_train), "· 검증", len(X_valid))

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("무조건 최빈 카테고리로 찍기: 정확도 %.4f" % dummy.score(X_valid, y_valid))

**약 0.18입니다.** 카테고리가 10개니 무작위로 찍으면 0.1, 최빈값으로 찍으면 0.18. 목표인 0.70은
이 기준선보다 한참 위이므로 **의미 있는 목표**입니다. (만약 어떤 카테고리가 전체의 80%였다면
"정확도 70%"는 아무것도 안 한 것보다 나쁜 성적이 됩니다. 기준선을 먼저 계산하는 이유입니다.)

---

## 4. 텍스트를 숫자로 — BoW와 TF-IDF

머신러닝 모델은 문자열을 먹지 못합니다. `"한결식품 얼큰 컵라면 110g"`을 숫자 배열로 바꿔야 합니다.
가장 기본적인 방법이 **BoW(Bag of Words, 단어 가방)** 입니다.

1. 전체 데이터에 나온 단어를 모아 **사전**을 만든다 (`얼큰`=0, `컵라면`=1, `한결식품`=2, ...)
2. 각 상품명을 **사전 길이만큼의 벡터**로 바꾸고, 등장한 단어 자리에 횟수를 적는다

이름 그대로 **단어를 가방에 쓸어 담는** 방식이라 어순은 사라집니다. "매운 치즈 라면"과 "치즈 매운 라면"이
같은 벡터가 됩니다. 상품명처럼 짧은 텍스트에서는 어순이 큰 의미가 없어서 이 단순함이 잘 통합니다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

예시 = [
    "한결식품 얼큰 컵라면 110g",
    "미소원 매운 라면 5개입",
    "포미 딸기 요거트 85g",
]

cv = CountVectorizer()
행렬 = cv.fit_transform(예시)

print("사전:", cv.get_feature_names_out())
print("\n행렬 크기:", 행렬.shape, "(문서 수 × 단어 수)")
pd.DataFrame(행렬.toarray(), columns=cv.get_feature_names_out(), index=["컵라면", "라면", "요거트"])

**`110g`와 `5개입`이 사라진 것을 보셨나요?** `CountVectorizer`의 기본 토큰 규칙(`token_pattern`)은
**두 글자 이상의 낱말**만 남기고, 순수한 숫자 토큰은 걸러냅니다. 즉 **토큰화 기준이 곧 피처 설계**입니다.

한국어에는 더 큰 문제가 있습니다. **공백 기준으로 자르면 조사가 붙은 채로 잘립니다.**
"라면이", "라면을", "라면은"이 전부 다른 단어가 됩니다. 제대로 하려면
[형태소 분석](https://github.com/karzit/temp/blob/master/glossary.md#morphological-analysis)(`kiwipiepy`, `konlpy` 등)이 필요합니다.
다행히 **상품명은 조사가 거의 없는 명사 나열**이라 공백 분리로도 꽤 버팁니다. 7절에서 형태소 분석 없이
같은 문제를 완화하는 방법(문자 n-gram)을 씁니다.

### TF-IDF — 흔한 단어의 힘을 빼기

BoW에는 약점이 있습니다. `프리미엄`, `대용량` 같은 마케팅 문구는 **모든 카테고리에 골고루** 나오는데,
등장 횟수만 세면 이런 단어도 큰 값을 갖습니다. [TF-IDF](https://github.com/karzit/temp/blob/master/glossary.md#tfidf)는
여기에 **"이 단어가 몇 개의 문서에 나오는가"** 로 벌점을 매깁니다.

```
TF-IDF(단어, 문서) = (문서 안 등장 횟수) × log(전체 문서 수 / 그 단어가 나온 문서 수)
```

**여러 문서에 두루 나오는 단어일수록 값이 작아집니다.** `컵라면`처럼 특정 카테고리에만 나오는 단어는
값이 커지고, `프리미엄`처럼 아무 데나 붙는 단어는 값이 작아집니다. 사람이 "이 단어가 중요하다"고
지정하지 않아도 **데이터가 알아서** 정하는 셈입니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

예시2 = 예시 + ["프리미엄 라면 특선", "프리미엄 요거트 세트"]

tv = TfidfVectorizer()
tfidf = tv.fit_transform(예시2)

pd.DataFrame(tfidf.toarray(), columns=tv.get_feature_names_out()).round(2)

마지막 두 행을 보면 **여러 문서에 나온 `프리미엄`의 값이, 같은 문서 안의 다른 단어보다 작습니다.**
반면 한 문서에만 나온 `특선`, `세트`는 값이 큽니다. 이것이 TF-IDF가 하는 일의 전부입니다.

---

## 5. 첫 모델 — TF-IDF + 로지스틱 회귀

[로지스틱 회귀](https://github.com/karzit/temp/blob/master/glossary.md#logistic-regression)는 텍스트 분류의 **표준 출발점**입니다.
단어 수천 개짜리 희소 행렬에서 잘 동작하고, 빠르고, 어떤 단어가 어느 카테고리를 밀어올렸는지 볼 수 있습니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
baseline.fit(X_train, y_train)

pred = baseline.predict(X_valid)
print("검증 정확도: %.4f" % accuracy_score(y_valid, pred))
print("사전 크기: %d 단어" % len(baseline.named_steps["tfidfvectorizer"].get_feature_names_out()))

**첫 시도에 목표(0.70)를 훌쩍 넘습니다.** 상품명은 카테고리를 알려주는 단어(`컵라면`, `요거트`)를
대놓고 포함하고 있어서, 텍스트 분류 중에서는 쉬운 편에 속합니다.

여기서 **`make_pipeline`을 쓴 이유**가 중요합니다. 벡터화와 모델을 하나로 묶으면
`fit`은 학습 데이터에만 적용되고, `predict` 때는 자동으로 `transform`만 호출됩니다.
직접 `TfidfVectorizer().fit_transform(전체_데이터)`를 부르면 **검증 데이터의 단어 분포가 사전에 섞여 들어가**
[데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage)이 됩니다. 표 데이터에서 스케일러를
분할 전에 `fit`하면 안 됐던 것과 정확히 같은 문제입니다.

어떤 단어가 어느 카테고리를 밀어올렸는지도 볼 수 있습니다.

In [ ]:
import numpy as np

vec = baseline.named_steps["tfidfvectorizer"]
clf = baseline.named_steps["logisticregression"]
words = vec.get_feature_names_out()

for i, cat in enumerate(clf.classes_[:4]):
    top = np.argsort(clf.coef_[i])[-6:][::-1]
    print(f"{cat:<12}", ", ".join(words[j] for j in top))

계수가 큰 단어가 **상식과 맞는지** 확인하는 습관을 들이세요. 표 데이터에서 변수중요도로
데이터 누출을 잡아냈던 것과 같은 점검입니다. 만약 여기서 `무료배송`이나 브랜드명이 상위에 올라온다면,
모델이 **상품이 아니라 판매자의 습관**을 배우고 있다는 뜻입니다.

---

## 6. 전처리는 정말 도움이 되는가

상품명에는 `[무료배송]`, `2+1`, `110g`, `_` 같은 것들이 섞여 있습니다.
"당연히 지우는 게 좋겠지"라고 생각하기 쉽습니다. **확인해봅시다.**

In [ ]:
import re


def clean_text(s):
    """상품명에서 판매 문구·숫자·특수문자를 걷어낸다."""
    s = re.sub(r"\[[^\]]*\]", " ", s)       # [무료배송] 같은 대괄호 블록
    s = re.sub(r"\([^)]*\)", " ", s)        # (대용량) 같은 괄호 블록
    s = re.sub(r"[0-9]+", " ", s)           # 숫자
    s = re.sub(r"[^가-힣A-Za-z ]", " ", s)  # 한글·영문·공백만 남기기
    return re.sub(r"\s+", " ", s).strip()   # 연속 공백 정리


for s in X_train.iloc[:5]:
    print(repr(s))
    print("   →", repr(clean_text(s)))

In [ ]:
X_train_c, X_valid_c = X_train.map(clean_text), X_valid.map(clean_text)

cleaned = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train_c, y_train)

print("정제 전: %.4f" % accuracy_score(y_valid, baseline.predict(X_valid)))
print("정제 후: %.4f" % accuracy_score(y_valid, cleaned.predict(X_valid_c)))

**성능이 오르지 않습니다. 오히려 조금 떨어집니다.**

당연해 보이는 전처리가 왜 도움이 안 됐을까요?

- **숫자와 단위도 신호였습니다.** `110g`은 라면 쪽, `1000ml`는 음료 쪽에 더 자주 붙습니다.
  숫자를 통째로 지우면서 그 신호가 사라졌습니다
- **TF-IDF가 이미 노이즈를 눌러줍니다.** `무료배송`은 모든 카테고리에 나오므로 IDF 가중치가 이미 낮습니다.
  손으로 지울 이유가 크지 않았던 것입니다
- 반면 잃은 것은 확실합니다. 정제는 **정보를 지우는 작업**이라, 지운 것 중에 쓸모 있는 것이 있으면 손해입니다

**전처리는 "당연히 하는 것"이 아니라 실험해서 채택하는 것입니다.** 다만 성능이 같다면
단순한 쪽(정제해서 사전이 작아진 쪽)을 고르기도 합니다. 이 데이터에서는 정제 없이 갑니다.

> 예외가 있습니다. **공백·대소문자 정리처럼 "같은 것을 같게 만드는" 정규화는 거의 항상 이득**입니다.
> `TfidfVectorizer`는 기본적으로 `lowercase=True`라 영문 대소문자는 이미 통일되어 있습니다.

---

## 7. 문자 n-gram — 형태소 분석 없이 한국어 다루기

공백으로 자르면 `치즈라면`과 `치즈 라면`이 서로 다른 단어가 됩니다. `요거트`와 `요구르트`도 남남입니다.
**단어 대신 글자 몇 개씩 잘라서 세면** 이 문제가 상당히 완화됩니다.

`analyzer="char_wb", ngram_range=(2, 3)`은 낱말 안에서 **글자 2~3개짜리 조각**을 만듭니다.

```
"컵라면"  →  "컵라", "라면", "컵라면"
```

`치즈라면`과 `치즈 라면`은 이제 `라면`이라는 조각을 공유합니다. 오타나 표기 흔들림에도 강해집니다.
대신 **피처 수가 크게 늘어납니다.**

In [ ]:
from sklearn.pipeline import make_union

configs = {
    "단어": TfidfVectorizer(),
    "문자 2~3": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    "단어+문자": make_union(
        TfidfVectorizer(),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    ),
}

for name, vectorizer_ in configs.items():
    model = make_pipeline(vectorizer_, LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(X_train, y_train)
    n_features = model[:-1].transform(X_train[:1]).shape[1]
    print(f"{name:<8} 정확도 {accuracy_score(y_valid, model.predict(X_valid)):.4f}  (피처 {n_features:,}개)")

**문자 n-gram이 조금 더 좋습니다.** 차이는 크지 않지만 피처 수는 몇 배로 늘었습니다.
이 데이터는 상품명의 띄어쓰기가 규칙적이라 얻는 것이 적었습니다. 띄어쓰기가 엉망인 실제
쇼핑몰 데이터라면 격차가 훨씬 커집니다.

**기억할 것:** 한국어 텍스트에서 형태소 분석기를 붙이기 전에 **문자 n-gram을 먼저 시도해보세요.**
설치도 필요 없고 인자 두 개면 됩니다.

---

## 8. 모델 바꿔보기

벡터화 방식을 정했으니 분류 모델을 비교합니다. 텍스트에서 자주 쓰이는 세 가지입니다.

| 모델 | 성격 |
|---|---|
| `LogisticRegression` | 표준 출발점. 안정적이고 계수 해석이 가능 |
| `MultinomialNB` (나이브 베이즈) | 단어 등장 확률을 곱하는 고전적 방법. **아주 빠름** |
| `LinearSVC` | 고차원 희소 데이터에서 강한 선형 SVM |

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC


def vectorizer():
    """이 노트북에서 채택한 벡터화 설정."""
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))


models = {
    "로지스틱 회귀": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "나이브 베이즈": MultinomialNB(),
    "선형 SVM": LinearSVC(random_state=RANDOM_STATE),
}

fitted = {}
for name, model in models.items():
    pipe = make_pipeline(vectorizer(), model).fit(X_train, y_train)
    fitted[name] = pipe
    print(f"{name:<10} 정확도 {accuracy_score(y_valid, pipe.predict(X_valid)):.4f}")

**세 모델의 차이가 크지 않습니다.** 이 정도 차이는 분할을 바꾸면 순위가 뒤집힐 수 있는 폭입니다.
텍스트 분류에서는 **모델을 바꾸는 것보다 피처(벡터화)를 바꾸는 것이 대개 더 큰 차이**를 만듭니다.

---

## 9. 정확도 하나로 끝내면 안 되는 이유

시험의 채점 기준은 정확도지만, **정확도는 소수 카테고리의 실패를 감춥니다.**
`시리얼/영양바`는 전체의 3%라, 이 카테고리를 통째로 틀려도 전체 정확도는 3%밖에 안 떨어집니다.

카테고리별로 나눠 봐야 합니다.

| 지표 | 뜻 | 언제 보나 |
|---|---|---|
| 정밀도(precision) | 이 카테고리라고 예측한 것 중 맞은 비율 | 잘못 넣는 비용이 클 때 |
| 재현율(recall) | 실제 이 카테고리 중 찾아낸 비율 | 놓치면 안 될 때 |
| f1-score | 둘의 조화평균 | 균형 |
| **macro avg** | **카테고리별 지표의 단순 평균** | **불균형 데이터의 진짜 성능** |
| weighted avg | 건수로 가중 평균한 값 | 정확도와 비슷하게 움직임 |

**macro 평균은 건수를 무시하고 카테고리를 동등하게 봅니다.** 3%짜리 카테고리를 못 맞히면
정확도는 멀쩡해도 macro f1은 뚝 떨어집니다.

In [ ]:
from sklearn.metrics import classification_report, f1_score

best = fitted["로지스틱 회귀"]
pred = best.predict(X_valid)

print(classification_report(y_valid, pred, zero_division=0))
print("macro f1: %.4f" % f1_score(y_valid, pred, average="macro"))

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(y.unique())
cm = confusion_matrix(y_valid, pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("예측")
ax.set_ylabel("실제")
ax.set_title("혼동 행렬 — 어디서 헷갈리는가")
plt.tight_layout()
plt.show()

**혼동 행렬은 대각선이 아니라 대각선 바깥을 보는 그림입니다.** 어떤 카테고리 쌍이 서로 넘어가는지 찾으세요.

- `즉석밥/간편식` ↔ `냉동식품`: `떡볶이`, `카레`처럼 **양쪽에 다 있는 상품**이 있습니다
- `유제품` ↔ `커피/차`: `라떼`가 양쪽에 걸칩니다
- 건수가 가장 적은 `시리얼/영양바`는 재현율이 전체 정확도보다 낮습니다 — **적게 배운 카테고리는 적게 예측되는 경향**이 있습니다

**헷갈리는 쌍이 데이터의 성격 때문인지, 모델의 부족 때문인지**를 구분하는 것이 다음 단계입니다.
그러려면 틀린 것을 직접 봐야 합니다.

---

## 10. 틀린 것을 직접 본다

**성능을 올리는 가장 확실한 방법은 오분류를 눈으로 읽는 것입니다.** 숫자만 보면
"91%구나"에서 끝나지만, 틀린 상품명 열몇 개를 읽으면 다음에 뭘 해야 할지 알게 됩니다.

In [ ]:
proba = best.predict_proba(X_valid)

오답 = pd.DataFrame({
    "상품명": X_valid.values,
    "실제": y_valid.values,
    "예측": pred,
    "확신도": proba.max(axis=1).round(2),
})
오답 = 오답[오답["실제"] != 오답["예측"]].sort_values("확신도", ascending=False)

print(f"검증 {len(X_valid)}건 중 오답 {len(오답)}건")
오답.head(15)

읽어보면 오답이 세 종류로 나뉩니다.

| 유형 | 예 | 고칠 수 있나 |
|---|---|---|
| **핵심어가 없는 상품명** | `산들바람 프리미엄 320g` | ✗ — 사람도 못 맞힙니다 |
| **양쪽에 다 쓰이는 단어** | `... 치즈스틱 ...`, `... 카레 ...` | △ — 브랜드·용량 같은 추가 단서가 필요 |
| **라벨 자체가 이상한 행** | 상품명은 명백한데 카테고리가 엉뚱 | ✗ — 데이터의 라벨 오류 |

**이 데이터에는 라벨 오류가 3% 섞여 있습니다**([생성 스크립트](https://github.com/karzit/temp/blob/master/notebooks/text-classification-practice/data/make_dataset.py)에서
일부러 넣었습니다). 실제 상품 데이터도 사람이 손으로 분류한 것이라 라벨 오류가 늘 있습니다.

여기서 중요한 결론이 나옵니다. **정확도 100%는 목표가 아닙니다.** 맞힐 수 없는 행이 섞여 있는데
훈련 데이터에서 100%를 만들면, 그것은 라벨 오류까지 통째로 외운
[과적합](https://github.com/karzit/temp/blob/master/glossary.md#overfitting)입니다. 확신도가 높은 오답부터 읽어보면
**모델이 자신 있게 틀리는 지점**이 보입니다.

---

## 11. 제출 파일 만들기 — 시험 형식 그대로

이제 시험이 요구하는 형태로 결과를 만듭니다. 여기서 점수가 갈리는 부분은 모델이 아니라 **형식**입니다.

**세 가지를 지킵니다.**

1. 최종 모델은 **훈련 데이터 전체**로 다시 학습합니다 (검증용으로 떼어놨던 20%도 이제 씁니다)
2. 예측 결과는 **훈련 데이터와 같은 컬럼 구조**여야 합니다 — `상품명`, `카테고리`
3. 파일명은 **`본인핸드폰번호_2.csv`** — 하이픈·언더바·공백 없이 숫자만 (`01012345678_2.csv`)

In [ ]:
import joblib

PHONE = "01012345678"  # 실제 시험에서는 본인 휴대폰 번호로 바꾸세요.

final_model = make_pipeline(
    vectorizer(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X, y)  # 분할하지 않은 전체 훈련 데이터

submission = test_x.copy()
submission["카테고리"] = final_model.predict(submission["상품명"])
submission.to_csv(f"{PHONE}_2.csv", index=False, encoding="utf-8-sig")

joblib.dump(final_model, f"{PHONE}_2.pkl")

print(submission.shape)
submission.head()

**확인 목록.**

- [ ] 행 수가 `02_test_x.csv`와 같은가 (1,500행) — 행을 지우거나 순서를 바꾸면 채점이 어긋납니다
- [ ] 컬럼이 `상품명`, `카테고리` 순인가
- [ ] 카테고리 값이 훈련 데이터에 있던 문자열 그대로인가 (숫자로 인코딩된 채 저장하는 실수가 흔합니다)
- [ ] 한글이 깨지지 않는가 — `encoding="utf-8-sig"`
- [ ] **노트북을 위에서부터 다시 실행해도 같은 결과가 나오는가** (채점 시 재현성을 확인합니다)

> **`test_x`에는 `dropna`를 하지 마세요.** 훈련 데이터에서는 결측 행을 지우는 게 맞지만,
> 테스트 데이터는 **모든 행에 대해 예측값을 내야** 합니다. 상품명이 비어 있으면 빈 문자열로 채워서라도
> 한 줄을 채웁니다. 행이 하나라도 빠지면 정답지와 줄이 어긋납니다.

이제 자가 채점입니다. **실제 시험에는 없는 단계**입니다(정답이 제공되지 않으니까요). 연습이니 확인해봅니다.

In [ ]:
test_y = pd.read_csv(os.path.join(DATA_DIR, "02_test_y.csv"))

acc = accuracy_score(test_y["카테고리"], submission["카테고리"])
print("테스트 정확도: %.4f" % acc)
print("macro f1     : %.4f" % f1_score(test_y["카테고리"], submission["카테고리"], average="macro"))
print("목표(0.70) 달성:", "예" if acc >= 0.70 else "아니오")

검증 정확도와 테스트 정확도가 비슷하게 나왔습니다. **떼어둔 검증 세트가 제 역할을 했다는 뜻**입니다.
두 숫자가 크게 벌어진다면 검증 방식이 잘못됐거나(누출) 두 데이터의 분포가 다른 것입니다.

---

## 정리

- **텍스트 분류는 "글자를 어떻게 숫자로 바꾸는가"가 절반**입니다. BoW → TF-IDF가 기본 경로입니다
- **TF-IDF는 흔한 단어의 힘을 자동으로 뺍니다.** 불용어 목록을 손으로 만들 필요가 크지 않습니다
- **기준선을 먼저 계산하세요.** 최빈 카테고리 0.18을 알아야 0.70이 의미 있는 목표인지 판단됩니다
- **벡터화와 모델은 `Pipeline`으로 묶습니다.** 사전을 전체 데이터로 만들면 데이터 누출입니다
- **전처리는 당연히 하는 것이 아니라 측정해서 채택합니다.** 이 데이터에서는 숫자를 지우자 오히려 떨어졌습니다
- **한국어에서는 문자 n-gram(`char_wb`)을 먼저 시도**하세요. 형태소 분석기 없이 띄어쓰기 흔들림을 흡수합니다
- **불균형 데이터에서는 macro f1을 함께** 봅니다. 정확도는 소수 카테고리의 실패를 감춥니다
- **오분류를 눈으로 읽으세요.** 고칠 수 있는 오답과 없는 오답(라벨 오류)이 구분됩니다
- **제출은 형식이 반**입니다. 행 수·컬럼 구조·파일명·재현성

## 스스로 확인해보기

- [ ] BoW와 TF-IDF의 차이를 한 문장으로 설명할 수 있다
- [ ] `TfidfVectorizer`를 전체 데이터에 `fit`하면 왜 문제인지 안다
- [ ] 클래스가 불균형할 때 정확도만 보면 안 되는 이유를 안다
- [ ] `analyzer="char_wb"`가 한국어에서 도움이 되는 이유를 설명할 수 있다
- [ ] 혼동 행렬에서 "어떤 두 카테고리가 서로 헷갈리는지" 읽을 수 있다
- [ ] 제출 파일이 갖춰야 할 조건 네 가지를 말할 수 있다

## 연습 문제

풀어본 뒤 [01_text_baseline_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline_solutions.ipynb)에서 확인하세요.

**문제 1.** 같은 상품명에 **서로 다른 카테고리**가 붙은 행이 훈련 데이터에 몇 건이나 있는지 세어보세요.
그런 행은 무엇을 뜻하며, 학습에서 빼는 것이 좋을까요?

**문제 2.** `TfidfVectorizer`의 `min_df`(너무 드문 단어 제거)와 `max_df`(너무 흔한 단어 제거)를
바꿔가며 검증 정확도와 피처 수를 비교하세요. `min_df=1, 2, 3`과 `max_df=1.0, 0.5`의 조합으로 충분합니다.
피처를 줄여도 성능이 유지된다면 무엇이 좋은가요?

**문제 3.** `LogisticRegression(class_weight="balanced")`로 학습해 소수 카테고리의 재현율이
어떻게 바뀌는지 `classification_report`로 비교하세요. 전체 정확도는 어떻게 되나요?
시험 기준이 정확도라면 이 옵션을 쓰는 것이 유리할까요?

**문제 4.** `GridSearchCV`로 벡터화 설정(`ngram_range`)과 로지스틱 회귀의 `C`를 함께 탐색하세요.
`Pipeline`의 파라미터 이름은 `단계이름__파라미터`(예: `tfidfvectorizer__ngram_range`) 형식입니다.

**문제 5.** 확신도(`predict_proba`의 최댓값)가 낮은 예측 100건을 뽑아, 그 구간의 정확도가
전체 정확도와 얼마나 다른지 확인하세요. 실무에서 이 숫자를 어디에 쓸 수 있을까요?

**문제 6.** 상품명에서 **브랜드(첫 단어)를 제거**하고 학습하면 정확도가 어떻게 되나요?
결과를 보고 "모델이 브랜드를 외우고 있었는지" 판단해보세요.

---

다음 노트북([02_keras_text](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text.ipynb))에서는
같은 문제를 Keras 신경망으로 풉니다. 시험이 제출물로 요구하는 `.h5` 모델 파일을 만들어보고,
**딥러닝이 이 문제에서 TF-IDF를 이기는지** 직접 확인합니다.